# Behavioral AML GNN (PayPal Topology)

This notebook demonstrates the **Behavioral GNN** approach designed for environments where counterparty information is missing (e.g., all transactions are to/from PayPal).

## 1. Introduction: Nodes & Features

Instead of linking Customers to specific beneficiaries, we link them to **Behavioral Nodes**. This transforms "similarity of action" into "graph connectivity".

### Node Types
- **`Customer`**: The main actors. Features include volume intensity and frequency.
- **`Bank`**: The funding source institution. Useful for catching rings targeting specific bank KYC weaknesses.
- **`Date`**: Absolute calendar dates. Captures synchronized mass-activation events.
- **`DayOfMonth`**: Days 1-31. Captures cyclical patterns like payroll or benefit fraud.
- **`DayOfWeek`**: Monday-Sunday. Captures operational schedules (e.g., "Friday Afternoon Structuring").
- **`AmountBin`**: Discretized transaction amounts. Specifically tuned to catch structuring (e.g., $9,900 range).

### Key Edge Types
- `(customer, active_on_date, date)`: Links users transacting on the same days.
- `(customer, transacts_vol, amount_bin)`: Links users using the same amount tactics.
- `(customer, uses_bank, bank)`: Links users operating from the same institution.

---

## 2. Parameter Tuning Guide

The effectiveness of the detection depends on these tunable parameters in the `config` and `model`:

### Graph Construction Parameters
- **`date_window`**: (Integer, e.g., 0, 1, 2). Controls "fuzzy" temporal matching. A window of 1 allows a Monday transaction to link to Tuesday nodes, catching near-miss synchronization.
- **`amount_bins`**: (List of floats). Defines the resolution of amount matching. Narrow bins around $10k help catch structuring; wider bins group general spending behaviors.

### Model & Detection Parameters
- **`architecture`**: ('sage' or 'gat'). 'sage' is robust and fast. 'gat' uses attention to weigh specific behaviors more heavily (good for complex evasion).
- **`hidden_channels`**: (Integer, e.g., 32, 64). Complexity of the learned embeddings. Higher values capture more nuance but require more data.
- **`epochs`**: (Integer). Training duration. Watch the loss curve; unsupervised GNNs typically converge quickly (10-30 epochs).
- **`contamination`**: (Float, 0.0 to 0.5). The expected percentage of anomalies in the data. Higher values flag more customers as "High Risk".

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add src to path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

from src.gnn.modeling.behavior_gnn import BehaviorGraphBuilder, BehavioralGNN, train_behavior_gnn, detect_behavioral_anomalies
from scripts.visualize_behavioral_graph import visualize_suspect

print("Modules loaded.")

## 3. Generate "Smurfing" Mock Data

We simulate a group of 20 Mules who all transact on **Fridays** (with some 1-day drift) with amounts between **$9,000 - $9,900**.

In [ ]:
np.random.seed(42)
n_normal = 500
n_mule = 20

# 1. Normal Data (Random days, Random amounts)
dates_normal = pd.date_range('2024-01-01', periods=n_normal*24, freq='h')
df_normal = pd.DataFrame({
    'cust_id': [f'User_{i}' for i in range(n_normal)],
    'amount': np.random.exponential(1000, n_normal),
    'date': np.random.choice(dates_normal, n_normal),
    'bank': np.random.choice(['Chase', 'Wells', 'BoA', 'Citi'], n_normal),
    'type': 'Normal'
})

# 2. Mule Data (Synchronized on Fridays/Saturdays, Structuring amounts)
fridays_fuzzy = ['2024-01-05', '2024-01-06', '2024-01-12', '2024-01-13']
df_mule = pd.DataFrame({
    'cust_id': [f'Mule_{i}' for i in range(n_mule)],
    'amount': np.random.uniform(9000, 9900, n_mule), 
    'date': np.random.choice(fridays_fuzzy, n_mule), 
    'bank': 'SketchyCreditUnion',                    
    'type': 'Mule'
})

df = pd.concat([df_normal, df_mule]).sample(frac=1).reset_index(drop=True)
print(f"Data Generated: {len(df)} txns. {n_mule} are Mules.")
df.tail()

## 4. Build Behavioral Graph

We use a **`date_window` of 1** to ensure that mules transacting on Friday and Saturday are linked to the same temporal context.

In [ ]:
config = {
    'customer': 'cust_id',
    'amount': 'amount',
    'date': 'date',
    'bank': 'bank',
    
    # --- Configurable Parameters --- 
    'date_window': 1, 
    'amount_bins': [-1, 1000, 5000, 9000, 10000, float('inf')] 
}

builder = BehaviorGraphBuilder(config)
data = builder.build(df)

print("Graph Structure:")
print(data)

## 5. Unsupervised Training

We can choose between **'sage'** (GraphSAGE) and **'gat'** (Graph Attention).

In [ ]:
model = BehavioralGNN(
    data.metadata(), 
    hidden_channels=32, 
    out_channels=32,
    architecture='gat' # Switch to 'sage' or 'gat'
 )

print("Training Behavioral GNN (Link Prediction objective)...")
model = train_behavior_gnn(model, data, epochs=20, lr=0.01)
print("Done.")

## 6. Anomaly Detection Results

In [ ]:
results = detect_behavioral_anomalies(model, data, builder.cust_map)

# Evaluate against Ground Truth
ground_truth = df[['cust_id', 'type']].drop_duplicates()
merged = results.merge(ground_truth, left_on='customer_id', right_on='cust_id')

print("Top 20 Highest Risk Customers:")
merged.sort_values('risk_score', ascending=False).head(20)

## 7. Visualization

We visualize the "Ego Graph" of the top suspect to understand *why* they were flagged. 
We look for clusters of Blue Nodes (Behaviors) connecting multiple Orange Nodes (Other Suspects).

In [ ]:
# Pick the #1 suspect
top_suspect = merged.sort_values('risk_score', ascending=False).iloc[0]['customer_id']

print(f"Visualizing Network for: {top_suspect}")

# Hops=2 shows the 'Ring' (other customers connected to the same behaviors)
visualize_suspect(top_suspect, data, builder, hops=2)